# #237 BJ-UNIFIED-POWER-FEATURE-SUPPORT-001 — causal power feature support (R2)

Rebuild timestamp-safe league/team/matchup strength features on the canonical
fixture identity. R2 rework addresses EV R1 C-01..C-04 + E-01/E-02.

**Contract**: BJ-UNIFIED-POWER-FEATURE-SUPPORT-001 / v1, `mode: candidate`
**Project**: BJ Data Platform · **Owner**: Engineer · **Parent**: #232
**Base**: #234 canonical source-schema interface (fixture identity consumed)

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists():
    ROOT = ROOT.parents[2]
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
import json
sys.path.insert(0, str(ROOT / "src"))
import bj237_power_lib as lib
print("ROOT:", ROOT)

ROOT: /home/drwho1985/beijing-lot-237-r2


## 1. Identity and coverage profiling (REQ-PW02, REQ-PW06)

In [2]:
matches = lib.load_xlsm_matches()
epochs = lib.load_decision_epochs(ROOT)
print("xlsm matches:", len(matches), "issues:", matches['issue'].nunique())
print("#203 epochs:", len(epochs), "issues")
line = pd.to_numeric(matches['handicap_line_norm'], errors='coerce')
print("xlsm lines null:", int(line.isna().sum()), "non-null:", int(line.notna().sum()))

xlsm matches: 195269 issues: 701
#203 epochs: 394 issues
xlsm lines null: 3282 non-null: 191987


### 1a. Support-loss waterfall (REQ-PW06, primitives only — C-02)

In [3]:
waterfall = pd.read_csv(ROOT / "results/bj237_power_feature_support_v2/support_loss_waterfall.csv")
display(waterfall)

,stage,label,rows,issues,reason
0,0.0,raw source rows,195269,701,NaN
1,1.0,identity ok (team ids + unique key),192703,701,NaN
2,2.0,league present,192703,701,NaN
3,3.0,decision epoch exists (#203),102243,344,NaN
4,NaN,NaN,2566,6,IDENTITY_MISSING_TEAM_ID
5,NaN,NaN,0,0,IDENTITY_DUPLICATE_ISSUE_MATCHIDX
6,NaN,NaN,0,0,IDENTITY_LEAGUE_BLANK
7,NaN,NaN,90460,357,NO_DECISION_EPOCH
8,NaN,NaN,1121,256,NOT_ELIGIBLE_HISTORY
9,NaN,NaN,101122,344,HISTORY_ELIGIBLE_SCORE_PRESENT


### 1b. #180 accepted P_POWER intersection independently reproduced (C-02)

In [4]:
rec = pd.read_csv(ROOT / "results/bj237_power_feature_support_v2/ppower_intersection_reconstruction.csv")
display(rec)
by_metric = dict(zip(rec['metric'], rec['value']))
print("published cells:", by_metric['published_cells'], "/ issues:", by_metric['published_issues'])
print("method:", by_metric['gate_method'])

,metric,value
0,gate_cells_total,357705
1,gate_issues_total,419
2,g2_vector_cells,4674
3,g2_vector_issues,189
4,g3_causal_epoch_cells,234
5,g3_causal_epoch_issues,9
6,g4_freeze_row_cells,234
7,g4_freeze_row_issues,9
8,g6_line_matched_cells,234
9,g6_line_matched_issues,9


published cells: 234 / issues: 9
method: primitives only: vector present + observed_at <= feature_cutoff_ts_utc + freeze triplet complete + exact non-null line equality; P_POWER_BJLINE_available flag NOT used


## 2. Causal feature builder output (REQ-PW01/03/04; C-01 fail-closed line)

In [5]:
feat = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v2/power_features.parquet")
print("cells:", len(feat), "issues:", feat['issue'].nunique())
print("triplet_ok:", int(feat['triplet_ok'].sum()), "line_ok:", int(feat['line_ok'].sum()))
# C-01: line_ok rows must have non-null grid line
line_null_ok = feat[feat['line_ok'] & feat['native_handicap_line'].isna()]
print("C-01 check — line_ok with NULL line:", len(line_null_ok), "(must be 0)")
not_line_null = feat[~feat['line_ok'] & feat['native_handicap_line'].notna()]
print("C-01 check — ~line_ok with line value:", len(not_line_null), "(must be 0)")

cells: 306720 issues: 344
triplet_ok: 305955 line_ok: 305955
C-01 check — line_ok with NULL line: 0 (must be 0)
C-01 check — ~line_ok with line value: 0 (must be 0)


### 2a. Publishable support by year (REQ-PW08, corrected — C-03)

In [6]:
pub = feat[feat['triplet_ok'] & feat['line_ok'] & feat['native_handicap_line'].notna() & ~feat['unseen_mask']]
by_year = pub.groupby('year').agg(cells=('outcome_code','count'),
                                  matches=('match_idx','nunique'),
                                  issues=('issue','nunique'))
display(by_year)
print("total cells:", len(pub), "issues:", pub['issue'].nunique())

,cells,matches,issues
year,,,
2019,52677,680,55
2020,35781,572,44
2021,56268,632,58
2022,50106,570,58
2023,46935,503,55
2024,48552,523,59
2025,14520,547,15


total cells: 304839 issues: 344


### 2b. Diagnostics: history depth, uncertainty, Elo, form, rest

In [7]:
diag = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v2/power_feature_diagnostics.parquet")
display(diag[['issue','fixture_id','identity_status','match_idx','league','home_team_id','away_team_id',
              'min_history_n','native_handicap_line','line_ok','triplet_ok','power_p3','power_p1','power_p0',
              'history_event_sha256']].head(6))

,issue,fixture_id,identity_status,match_idx,league,home_team_id,away_team_id,min_history_n,native_handicap_line,line_ok,triplet_ok,power_p3,power_p1,power_p0,history_event_sha256
0,190101,fix_190101_1_f821d5d7dd16042d,CANONICAL,1,澳洲甲,2910,2913,127,1.0,True,True,0.502418,0.235119,0.262463,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...
1,190101,fix_190101_2_3adf1c1a32b740ea,CANONICAL,2,以超,1607,1659,128,0.0,True,True,0.405932,0.289509,0.304559,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...
2,190101,fix_190101_3_8a7b8410ce3bb26c,CANONICAL,3,澳洲甲,20394,14549,130,0.0,True,True,0.344079,0.283913,0.372008,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...
3,190101,fix_190101_4_eb6f5c5b4074153c,CANONICAL,4,英超,31,59,225,0.0,True,True,0.438178,0.293833,0.267989,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...
4,190101,fix_190101_5_94573b398b16dcb7,CANONICAL,5,英乙,1193,1199,104,0.0,True,True,0.362143,0.318461,0.319396,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...
5,190101,fix_190101_6_f4279c0500884173,CANONICAL,6,英超,19,29,243,-2.0,True,True,0.161940,0.185351,0.652708,7586255f5a7d1c67a0160d2c506b94e1512c4bd837b140...


## 3. Support and calibration by issue/year/league/line (REQ-PW07, C-03)

In [8]:
sup = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v2/support_summary.parquet")
print("support groups (issue/year/league/line):", len(sup))
print("rows sum:", int(sup['rows'].sum()))
display(sup.head(12))

support groups (issue/year/league/line): 33142
rows sum: 304839


,issue,year,league,native_handicap_line,rows,issues,matches
0,190101,2019,亚洲杯,-2.0,12,1,4
1,190101,2019,亚洲杯,-1.0,9,1,3
2,190101,2019,以超,-1.0,3,1,1
3,190101,2019,以超,0.0,21,1,7
4,190101,2019,墨西联,-1.0,6,1,2
5,190101,2019,墨西联,0.0,18,1,6
6,190101,2019,法国杯,-2.0,3,1,1
7,190101,2019,法国杯,-1.0,12,1,4
8,190101,2019,法国杯,0.0,12,1,4
9,190101,2019,澳洲甲,-2.0,3,1,1


In [9]:
cal = pd.read_parquet(ROOT / "results/bj237_power_feature_support_v2/calibration_through_2023.parquet")
print("calibration groups (<=2023):", len(cal))
if len(cal):
    display(cal.head(12))
    print("calibration n sum:", int(cal['n'].sum()))

calibration groups (<=2023): 78012


,issue,year,league,native_handicap_line,outcome_code,n,mean_pred,mean_actual
0,190101,2019,亚洲杯,-2.0,0,4,0.722797,0.750000
1,190101,2019,亚洲杯,-2.0,1,4,0.161929,0.000000
2,190101,2019,亚洲杯,-2.0,3,4,0.115274,0.250000
3,190101,2019,亚洲杯,-1.0,0,3,0.593379,1.000000
4,190101,2019,亚洲杯,-1.0,1,3,0.234937,0.000000
5,190101,2019,亚洲杯,-1.0,3,3,0.171684,0.000000
6,190101,2019,以超,-1.0,0,1,0.435321,0.000000
7,190101,2019,以超,-1.0,1,1,0.264410,0.000000
8,190101,2019,以超,-1.0,3,1,0.300270,1.000000
9,190101,2019,以超,0.0,0,7,0.329907,0.142857


calibration n sum: 239034


## 4. Internal vs external data contributions (REQ-PW05)

In [10]:
ext = pd.read_csv(ROOT / "results/bj237_power_feature_support_v2/external_football_data_census.csv")
print("external football-data files:", len(ext), "rows:", int(ext['rows'].sum()))
display(ext.head(8))

external football-data files: 25 rows: 8982


,file,rows,competition,first_date,last_date,error
0,2021_D1.csv,306,D1,NaN,NaN,NaN
1,2021_E0.csv,380,E0,NaN,NaN,NaN
2,2021_F1.csv,380,F1,NaN,NaN,NaN
3,2021_I1.csv,380,I1,NaN,NaN,NaN
4,2021_SP1.csv,380,SP1,NaN,NaN,NaN
5,2122_D1.csv,306,D1,NaN,NaN,NaN
6,2122_E0.csv,380,E0,NaN,NaN,NaN
7,2122_F1.csv,380,F1,NaN,NaN,NaN


## 5. Sealed-label policy verification (REQ-PW07)

In [11]:
assert cal['year'].max() <= 2023, "calibration must never open 2024+"
print("OK: calibration table uses only <=2023 labels")

OK: calibration table uses only <=2023 labels


## 6. Package verification (incl. 13 remanifested tampers)

In [12]:
import bj237_verify_package as vf
defects = vf.verify(ROOT / "results/bj237_power_feature_support_v2", ROOT)
print("verify defects:", defects if defects else "NONE — PACKAGE VERIFIED OK")
gaps = vf.tamper_attacks(ROOT / "results/bj237_power_feature_support_v2", ROOT)
print("tamper gaps:", gaps if gaps else "NONE — 13/13 attacks detected")

verify defects: NONE — PACKAGE VERIFIED OK


tamper gaps: NONE — 13/13 attacks detected


## Conclusion (REQ-PW08)

Positive timestamp-safe power support: **304,839 causal publishable cells / 344
issues** (corrected; EV R1 headline). REQ-PW06 primitive reconstruction:
**234 cells / 9 issues** for the accepted #180 baseline. Aggregates keyed by
(issue, year, league, line); calibration ≤ 2023 only; 13/13 tamper attacks
detected; package published atomically (E-02).